基于指数退避算法，设置工具调用失败时的重试策略。

jitter是为了避免大量工具的重试请求集中在固定的时间点，引入抖动。

假设：按照策略，两次工具调用请求的时间间隔应为10秒，加入抖动后，可能为8.9秒，也可能为10.2秒。

模型的初始化

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)


举例：带抖动

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
import datetime

def write_times(s):
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")

count = 1
start_time = None

@tool
def get_weather(city: str):
    """
    查询指定城市的天气

    Args:
        city: 城市名称

    Returns:
        None
    """
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        #计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第{count}次调用，当前时间：{start_time}，和上次调用时间间隔{interval}秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")

agent = create_agent(
    model=main_model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware 用于捕获工具执行中的异常并自动重试
        ToolRetryMiddleware(
            max_retries=6, #最大重试次数（不包含初始的那次调用，一共最多调用 1 + 6 = 7 次）
            backoff_factor=2.0, # 指数退避因子（每次重试等待时间乘以2）
            initial_delay=1.0, # 第一次重试前的初始等待时间（1秒）
            max_delay=10.0, # 最大等待延迟上限（防止指数增长无限大，限制在 10秒）
            jitter=True, # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
            retry_on=(TimeoutError,), # 仅针对捕获到特定的TimeoutError 异常时才触发重试
            on_failure="continue" # 当达到最大重试数依然失败时， Agent的行为：“continue”表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage(content="今天深圳天气如何？")],
})

# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最红给出的兜底回复
for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

今天深圳天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_4795735b6234497a9811f54d)
 Call ID: call_4795735b6234497a9811f54d
  Args:
    city: 深圳
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

抱歉，由于系统故障，我无法为您查询到深圳今天的天气情况。建议您通过其他天气应用或网站查看最新预报。
